# NWEA Categories

The purpose of this notebook is to take the RIT scores that students earn on the different categories (goals) on NWEA and convert them to a percentage. The chart from the NWEA technical manual will be used but this only shows RIT scores by every five percentile so the performance will only be able to be reflected to the near fifth percentile.

In [ ]:
import numpy as np
import pandas as pd

## Loading Data

The roster has to be obtained from the Aeries query that is given. NWEA results need to be the raw data file from the NWEA site. The percentile charts were taken from the technical manual for NWEA. They can be obtained through a pdf upload in excel but have to have a separate file generated for each subject and the time of year the test was taken. The percentiles of 1 and 99 percent have to be created rows based on the lowest possible score (100) and the highest possible score (350). 

In [ ]:
# LIST STU SC ID NM GR LF SPECIALED 
roster = pd.read_excel(r"C:\Users\derek.castleman\OneDrive - Wonderful College Prep Academy\Desktop\Roster NWEA.xlsx")

fall_math = pd.read_excel(r"C:\Users\derek.castleman\OneDrive - Wonderful College Prep Academy\Desktop\NWEA Categories\Fall Math.xlsx")
fall_ela = pd.read_excel(r"C:\Users\derek.castleman\OneDrive - Wonderful College Prep Academy\Desktop\NWEA Categories\Fall ELA.xlsx")
winter_math = pd.read_excel(r"C:\Users\derek.castleman\OneDrive - Wonderful College Prep Academy\Desktop\NWEA Categories\Winter Math.xlsx")
winter_ela = pd.read_excel(r"C:\Users\derek.castleman\OneDrive - Wonderful College Prep Academy\Desktop\NWEA Categories\Winter ELA.xlsx")

nwea = pd.read_csv(r"C:\Users\derek.castleman\OneDrive - Wonderful College Prep Academy\Desktop\AssessmentResults.csv")

In [ ]:
roster

### Fixing Charts

First an input will be asked in which you will have to choose the time of year for the tests, allowing for the right charts to be selected. The charts will then be melted to turn all the grades and their associated scores into columns for each associated percentile.

In [ ]:
a = input("What test are you interested in: Fall or Winter?") #Input the time of year of interest
a

In [ ]:
# Takes the input and runs through an if then statement to find the correct charts
if a == "Fall":
    nwea_math = fall_math
    nwea_ela = fall_ela
else:
    nwea_math = winter_math
    nwea_ela = winter_ela

In [ ]:
nwea_math

In [ ]:
nwea_ela

In [ ]:
# Melts the math chart for grades with a value for score
math_melt = nwea_math.melt(id_vars=["Pct"], var_name="Grade", value_name="Score")
math_melt

In [ ]:
# Melts the ELA chart similar to Math
ela_melt = nwea_ela.melt(id_vars=["Pct"], var_name="Grade", value_name="Score")
ela_melt

## NWEA Cleaning

All that happens in this section is that the rows that are False for a growth measure are eliminated to leave only the results that earned a Growth measure in MAP. This will eliminate students who took duplicate tests in the process. Then two dataframes are created with one being for Math and the other being for ELA. Then the columns that are associated with the first four claims are selected since these are the only ones that have data.

In [ ]:
nwea

In [ ]:
# Selects rows that are not false for Growth Measure
nwea = nwea[nwea['GrowthMeasureYN'] != False]
nwea

In [ ]:
# Selects Math
math = nwea[nwea['Subject'] == 'Mathematics']
math

In [ ]:
# Selects ELA
ela = nwea[nwea['Subject'] == 'Language Arts']
ela

In [ ]:
# Selects the columns of interest in Math
math_columns = math.iloc[:, [5, 7, 77, 78, 81, 82, 83, 86, 87, 88, 91, 92, 93, 96]]
math_columns

In [ ]:
# Selects the columns of interest in ELA
ela_columns = ela.iloc[:, [5, 7, 77, 78, 81, 82, 83, 86, 87, 88, 91, 92, 93, 96]]
ela_columns

### Roster Cleanup

The issue with the roster is that students in kindergarten are listed as 0 so this value is changed with K.

In [ ]:
roster.info()

In [ ]:
roster['Grade'] = roster['Grade'].replace(0, 'K')
roster.info()

In [ ]:
roster

### Math

The first thing will be that the Math is merged with the student roster. Then each claim (one through four) will be selected by choosing the columns that are associated with, grabbing the claim name, level and RIT score that is associated with it. Then each claim will be merged with the Math chart. For the merge the function merge_asof will be used allowing the RIT score to match with the closest percentage but if there is a tie between percentages then the lowest one will be selected due to the restriction of the function.  When all the claims have been matched they will then all be merged to create one data frame with them all.

In [ ]:
# Merges Math MAP results with the roster
math = pd.merge(roster, math_columns, how='inner', left_on='Student ID', right_on='StudentID')
math

In [ ]:
# Selects the data for claim one
claim_one = math.iloc[:, [0, 1, 2, 3, 4, 5, 7, 8, 9, 10]]
claim_one

In [ ]:
# Selects the columns for claim two
claim_two = math.iloc[:, [0, 1, 2, 3, 4, 5, 7, 11, 12, 13]]
claim_two

In [ ]:
# Selects the columns for claim three
claim_three = math.iloc[:, [0, 1, 2, 3, 4, 5, 7, 14, 15, 16]]
claim_three

In [ ]:
# Selects the columns for claim four
claim_four = math.iloc[:, [0, 1, 2, 3, 4, 5, 7, 17, 18, 19]]
claim_four

In [ ]:
# Changes the data type for score
math_melt['Score'] = math_melt['Score'].astype('Int64')
math_melt.info()

In [ ]:
claim_one.info()

In [ ]:
claim_one['Goal1RitScore'] = claim_one['Goal1RitScore'].astype('Int64') #Sets RIT score to same datatype
claim_one = claim_one.dropna(subset=['Goal1RitScore']) # Drops any rows with no RIT score
claim_one = claim_one.sort_values(['Goal1RitScore', 'Grade']) # Sorts values for asof function
math_melt = math_melt.sort_values(['Score', 'Grade']) # Sorts math chart for the function.
# Merges both dataframes on score and selects the nearest Math chart score to Claim RIT score
claim_one = pd.merge_asof(claim_one, math_melt, by='Grade', left_on='Goal1RitScore', right_on = 'Score', direction='nearest')
claim_one

In [ ]:
claim_one.info()

In [ ]:
claim_two['Goal2RitScore'] = claim_two['Goal2RitScore'].astype('Int64')
claim_two = claim_two.dropna(subset=['Goal2RitScore'])
claim_two = claim_two.sort_values(['Goal2RitScore', 'Grade'])
math_melt = math_melt.sort_values(['Score', 'Grade'])
claim_two = pd.merge_asof(claim_two, math_melt, by='Grade', left_on='Goal2RitScore', right_on = 'Score', direction='nearest')
claim_two

In [ ]:
claim_three['Goal3RitScore'] = claim_three['Goal3RitScore'].astype('Int64')
claim_three = claim_three.dropna(subset=['Goal3RitScore'])
claim_three = claim_three.sort_values(['Goal3RitScore', 'Grade'])
math_melt = math_melt.sort_values(['Score', 'Grade'])
claim_three = pd.merge_asof(claim_three, math_melt, by='Grade', left_on='Goal3RitScore', right_on = 'Score', direction='nearest')
claim_three

In [ ]:
claim_three.info()

In [ ]:
claim_four['Goal4RitScore'] = claim_four['Goal4RitScore'].astype('Int64')
claim_four = claim_four.dropna(subset=['Goal4RitScore'])
claim_four = claim_four.sort_values(['Goal4RitScore', 'Grade'])
math_melt = math_melt.sort_values(['Score', 'Grade'])
claim_four = pd.merge_asof(claim_four, math_melt, by='Grade', left_on='Goal4RitScore', right_on = 'Score', direction='nearest')
claim_four

In [ ]:
# Changes the PCT column to Goal One PCT and drops the extra score from the Math chart
claim_one = claim_one.rename(columns={
    'Pct': 'Goal One PCT'})
claim_one = claim_one.drop('Score', axis=1)
claim_one

In [ ]:
claim_two = claim_two.rename(columns={
    'Pct': 'Goal Two PCT'})
claim_two = claim_two.drop('Score', axis=1)
claim_two

In [ ]:
claim_three = claim_three.rename(columns={
    'Pct': 'Goal Three PCT'})
claim_three = claim_three.drop('Score', axis=1)
claim_three

In [ ]:
claim_four = claim_four.rename(columns={
    'Pct': 'Goal Four PCT'})
claim_four = claim_four.drop('Score', axis=1)
claim_four

In [ ]:
# Merge claim one with claim two to start rebuilding Math data frame
math = pd.merge(claim_one, claim_two, how='outer', on=['School', 'Student ID', 'Student Name', 'Grade', 'LangFlu', 'SPECIALED Value', 'Subject'])
math

In [ ]:
math = pd.merge(math, claim_three, how='outer', on=['School', 'Student ID', 'Student Name', 'Grade', 'LangFlu', 'SPECIALED Value', 'Subject'])
math

In [ ]:
math = pd.merge(math, claim_four, how='outer', on=['School', 'Student ID', 'Student Name', 'Grade', 'LangFlu', 'SPECIALED Value', 'Subject'])
math

### ELA

Everything that was done with the Math portion of the test will be duplicated once again but now for the ELA portion of the test.

In [ ]:
ela = pd.merge(roster, ela_columns, how='inner', left_on='Student ID', right_on='StudentID')
ela

In [ ]:
claim_one = ela.iloc[:, [0, 1, 2, 3, 4, 5, 7, 8, 9, 10]]
claim_one

In [ ]:
claim_two = ela.iloc[:, [0, 1, 2, 3, 4, 5, 7, 11, 12, 13]]
claim_two

In [ ]:
claim_three = ela.iloc[:, [0, 1, 2, 3, 4, 5, 7, 14, 15, 16]]
claim_three

In [ ]:
claim_four = ela.iloc[:, [0, 1, 2, 3, 4, 5, 7, 17, 18, 19]]
claim_four

In [ ]:
ela_melt['Score'] = ela_melt['Score'].astype('Int64')
ela_melt.info()

In [ ]:
claim_one.info()

In [ ]:
claim_one['Goal1RitScore'] = claim_one['Goal1RitScore'].astype('Int64')
claim_one = claim_one.dropna(subset=['Goal1RitScore'])
claim_one = claim_one.sort_values(['Goal1RitScore', 'Grade'])
ela_melt = ela_melt.sort_values(['Score', 'Grade'])
claim_one = pd.merge_asof(claim_one, ela_melt, by='Grade', left_on='Goal1RitScore', right_on = 'Score', direction='nearest')
claim_one

In [ ]:
claim_one.info()

In [ ]:
claim_two['Goal2RitScore'] = claim_two['Goal2RitScore'].astype('Int64')
claim_two = claim_two.dropna(subset=['Goal2RitScore'])
claim_two = claim_two.sort_values(['Goal2RitScore', 'Grade'])
ela_melt = ela_melt.sort_values(['Score', 'Grade'])
claim_two = pd.merge_asof(claim_two, ela_melt, by='Grade', left_on='Goal2RitScore', right_on = 'Score', direction='nearest')
claim_two

In [ ]:
claim_three['Goal3RitScore'] = claim_three['Goal3RitScore'].astype('Int64')
claim_three = claim_three.dropna(subset=['Goal3RitScore'])
claim_three = claim_three.sort_values(['Goal3RitScore', 'Grade'])
ela_melt = ela_melt.sort_values(['Score', 'Grade'])
claim_three = pd.merge_asof(claim_three, ela_melt, by='Grade', left_on='Goal3RitScore', right_on = 'Score', direction='nearest')
claim_three

In [ ]:
claim_three.info()

In [ ]:
claim_four['Goal4RitScore'] = claim_four['Goal4RitScore'].astype('Int64')
claim_four = claim_four.dropna(subset=['Goal4RitScore'])
claim_four = claim_four.sort_values(['Goal4RitScore', 'Grade'])
ela_melt = ela_melt.sort_values(['Score', 'Grade'])
claim_four = pd.merge_asof(claim_four, ela_melt, by='Grade', left_on='Goal4RitScore', right_on = 'Score', direction='nearest')
claim_four

In [ ]:
claim_one = claim_one.rename(columns={
    'Pct': 'Goal One PCT'})
claim_one = claim_one.drop('Score', axis=1)
claim_one

In [ ]:
claim_two = claim_two.rename(columns={
    'Pct': 'Goal Two PCT'})
claim_two = claim_two.drop('Score', axis=1)
claim_two

In [ ]:
claim_three = claim_three.rename(columns={
    'Pct': 'Goal Three PCT'})
claim_three = claim_three.drop('Score', axis=1)
claim_three

In [ ]:
claim_four = claim_four.rename(columns={
    'Pct': 'Goal Four PCT'})
claim_four = claim_four.drop('Score', axis=1)
claim_four

In [ ]:
ela = pd.merge(claim_one, claim_two, how='outer', on=['School', 'Student ID', 'Student Name', 'Grade', 'LangFlu', 'SPECIALED Value', 'Subject'])
ela

In [ ]:
ela = pd.merge(ela, claim_three, how='outer', on=['School', 'Student ID', 'Student Name', 'Grade', 'LangFlu', 'SPECIALED Value', 'Subject'])
ela

In [ ]:
ela = pd.merge(ela, claim_four, how='outer', on=['School', 'Student ID', 'Student Name', 'Grade', 'LangFlu', 'SPECIALED Value', 'Subject'])
ela

### Finishing Up

To generate one final dataframe, the math dataframe and the ela dataframe are concatenated. Then a CVS file is generated that will allow for download.

In [ ]:
final = pd.concat([math, ela])
final

In [ ]:
import base64
from IPython.display import HTML

def create_download_link( df, title = "NWEA Categories", filename = "NWEA Categories"):
    csv = df.to_csv(index=False)
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

create_download_link(final)